In [1]:
import time
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cvxpy as cp
import os

In [2]:
import tempfile

def atomic_pickle_dump(obj, final_path, verbose=True):
    dir_name = os.path.dirname(final_path)

    if verbose:
        print(f"[atomic_pickle_dump] Saving to temporary file in: {dir_name}")

    with tempfile.NamedTemporaryFile(
        dir=dir_name, delete=False, suffix=".tmp"
    ) as tmp:
        pickle.dump(obj, tmp)
        tmp.flush()
        os.fsync(tmp.fileno())
        tmp_path = tmp.name

        if verbose:
            print(f"[atomic_pickle_dump] ✓ Written and flushed temp file: {tmp_path}")

    os.replace(tmp_path, final_path)

    if verbose:
        print(f"[atomic_pickle_dump] ✅ Successfully promoted temp file → {final_path}")


# 2. SPDS Algorithm with Constraint
Result savepath: /content/drive/MyDrive/experiment_result/

In [9]:
# --- Hàm xây dựng Ma trận A (từ Ảnh 2) ---
def build_A_matrix(n_goods, n_slots):
    """Tạo ma trận ánh xạ Goods -> Slots"""
    A = np.zeros((n_slots, n_goods))
    for j in range(n_goods):
        A[j % n_slots, j] = 1.0
    return A

# --- Hàm buyer best response for each user used in SPDS
def buyer_best_response_cvx(v_i, p, A, q_i, B_i, utility_type="Linear"):
    """
    Phiên bản siêu tốc của buyer_best_response.
    Sử dụng Closed-form solution cho mọi hàm utility.
    KHÔNG DÙNG CVXPY.
    """
    # 1. Tính giá hiệu dụng (Effective Price)
    # p: (M,), A.T @ q_i: (M,)
    effective_price = p + (A.T @ q_i)

    # An toàn: Đảm bảo giá > 0 để tránh chia cho 0
    safe_price = np.maximum(effective_price, 1e-9)

    m_goods = len(v_i)
    x = np.zeros(m_goods)

    # =========================================================
    # 1. LINEAR UTILITY: U = sum(v * x)
    # =========================================================
    if utility_type == "Linear":
        # Chiến thuật: Bang-per-buck (Mua tất tay món hời nhất)
        bang_per_buck = v_i / safe_price
        best_idx = np.argmax(bang_per_buck)
        x[best_idx] = B_i / safe_price[best_idx]

    # =========================================================
    # 2. COBB-DOUGLAS: U = prod(x^alpha) hoặc sum(alpha * log(x))
    # =========================================================
    elif utility_type == "Cobb-Douglas":
        # Chuẩn hóa alpha (Bắt buộc để thỏa mãn Budget constraint)
        sum_v = np.sum(v_i)
        if sum_v > 0:
            alpha = v_i / sum_v
        else:
            alpha = np.zeros_like(v_i) # Tránh lỗi nếu v toàn 0

        # Công thức: Chi tiêu đúng tỷ lệ alpha
        # x_i = (alpha_i * Budget) / Price_i
        x = (alpha * B_i) / safe_price

    # =========================================================
    # 3. LEONTIEF: U = min(x / v)
    # =========================================================
    elif utility_type == "Leontief":
        # Chiến thuật: Mua theo tỷ lệ cố định của v
        # Giá của 1 combo chuẩn = sum(v_i * p_i)
        cost_of_one_bundle = np.dot(v_i, safe_price)

        if cost_of_one_bundle > 0:
            # Số lượng combo mua được
            num_bundles = B_i / cost_of_one_bundle
            x = num_bundles * v_i
        else:
            x = np.zeros(m_goods)

    # =========================================================
    # 4. CES UTILITY: U = (sum v * x^rho)^(1/rho)
    # =========================================================
    elif utility_type == "CES_8":
        # CẤU HÌNH rho (m) TẠI ĐÂY
        # Trong code cũ bạn để m = 1/2
        rho = 0.5

        # --- CASE A: CONCAVE (rho < 1, rho != 0) ---
        # Đây là trường hợp thay thế (Substitute) -> Mua nhiều loại
        if rho < 1:
            # Tính Sigma (Elasticity of Substitution)
            # sigma = 1 / (1 - rho)
            sigma = 1.0 / (1.0 - rho)

            # Tính phần tử tỷ lệ: Term_i = (v_i / p_i)^sigma
            # Dùng np.maximum cho v_i để tránh v=0 gây lỗi log hoặc mũ âm
            term = np.power(v_i / safe_price, sigma)

            # Tính mẫu số chung: Sum (p_j * term_j)
            denom = np.dot(safe_price, term)

            if denom > 0:
                # x_i = (B * term_i) / denom
                x = (B_i * term) / denom
            else:
                 # Fallback nếu v=0 hết
                 x = np.zeros(m_goods)

        # --- CASE B: CONVEX (rho > 1) ---
        # Đây là trường hợp "Winner Takes All" giống Linear
        else:
            # So sánh tỷ lệ: v^(1/rho) / p
            v_transformed = np.power(v_i, 1.0/rho)
            bang_per_buck = v_transformed / safe_price

            best_idx = np.argmax(bang_per_buck)
            x[best_idx] = B_i / safe_price[best_idx]

    return x

# --- GROUND TRUTH OPTIMAL RESULT
def solve_centralized_optimal_fair(valuations, budgets, supply_s, b_mat, A, utility_type="Linear"):
    """
    Returns:
        optimal_value (float): Giá trị hàm mục tiêu tối ưu.
        optimal_X (np.ndarray): Ma trận phân bổ tối ưu (N x M).
    """
    n, m = valuations.shape
    X = cp.Variable((n, m), nonneg=True)

    constraints = [
        cp.sum(X, axis=0) <= supply_s
    ]
    for i in range(n):
        constraints.append(A @ X[i] <= b_mat[i])

    # --- XÂY DỰNG OBJECTIVE ---
    if utility_type == "Linear":
        utilities = cp.sum(cp.multiply(valuations, X), axis=1)
        primal_utility = cp.sum(cp.multiply(budgets, cp.log(utilities + 1e-12)))

    elif utility_type == "Cobb-Douglas":
        eps = 1e-12
        alpha = valuations / (np.sum(valuations, axis=1, keepdims=True))
        log_utilities = cp.sum(cp.multiply(alpha, cp.log(X + eps)), axis=1)
        primal_utility = cp.sum(cp.multiply(budgets, log_utilities))

    elif utility_type == "Leontief":
        # Lưu ý: Leontief trong CVXPY có thể phức tạp/chậm với quy mô lớn
        inv_valuations = np.divide(
            1.0,
            valuations,
            out=np.full_like(valuations, 1e30),
            where=(valuations > 0)
        )

        # 2. Nhân X với ma trận nghịch đảo hằng số này
        # ratio_matrix[i, j] = X[i, j] * (1 / v[i, j])
        weighted_X = cp.multiply(X, inv_valuations)

        # 3. Lấy min theo hàng
        utilities = cp.min(weighted_X, axis=1)

        # 4. Tính hàm mục tiêu
        primal_utility = cp.sum(cp.multiply(budgets, cp.log(utilities + 1e-12)))

    elif utility_type == "CES_8":
        # Nếu chạy Solver tập trung, m NÊN nhỏ hơn hoặc bằng 1 (ví dụ 0.5 hoặc -1).
        # Nếu đặt m = 8, Solver ECOS/SCS sẽ KHÔNG giải được (báo lỗi DCPError).
        eps = 1e-9
        m_ces = 1/2

        # Công thức: log_util = (1/m) * log( sum( v * x^m ) )
        # Tính tổng trọng số lũy thừa theo hàng (axis=1)
        inner_term = cp.sum(cp.multiply(valuations, cp.power(X + eps, m_ces)), axis=1)

        # Logarit hóa hàm mục tiêu
        log_utilities = (1.0 / m_ces) * cp.log(inner_term)

        primal_utility = cp.sum(cp.multiply(budgets, log_utilities))

    objective = cp.Maximize(primal_utility)
    prob = cp.Problem(objective, constraints)

    print(f"--- Solving Centralized Fair Problem ({utility_type}) ---")
    try:
        prob.solve(solver=cp.ECOS, verbose=False)
    except:
        prob.solve(solver=cp.SCS, verbose=False)

    # TRẢ VỀ: (Giá trị mục tiêu, Ma trận X tối ưu)
    return prob.value, X.value

In [10]:
""" UTILITY FUNCTIONS """
# --- CES / Linear (m = CES exponent, m=1 → Linear, m='inf' → Leontief-like max) ---
def calculate_ces_utility(allocation_vec, valuation_vec, m_ces = 1/2):
    eps = 1e-9
    return np.power(np.power(allocation_vec + eps, m_ces).T @ valuation_vec, (1/m_ces))

# --- COBB–DOUGLAS UTILITY ---
def calculate_cd_utility(allocation_vec, valuations_vec):
    eps = 1e-12
    v = np.atleast_2d(valuations_vec)
    x = np.atleast_2d(allocation_vec)

    # Normalize weights α_j = v_j / sum(v)
    weights = v / (np.sum(v, axis=1, keepdims=True))

    # log utility = Σ α_j log(x_j)
    log_u = np.sum(weights * np.log(x + eps), axis=1)

    u = np.exp(log_u)

    return u

# --- LEONTIEF UTILITY ---
def calculate_leo_utility(allocation_vec, valuations_vec):
    v = np.atleast_2d(valuations_vec)
    x = np.atleast_2d(allocation_vec)
    # 1. Khởi tạo mảng kết quả là Vô cực (Infinity)
    # Để nếu v=0, giá trị tại đó vẫn là Inf và không bị hàm min chọn phải
    ratios = np.full_like(x, 1e30)

    # 2. Chỉ thực hiện phép chia ở những nơi v > 0
    # Ghi đè kết quả phép chia vào mảng ratios
    np.divide(x, v, out=ratios, where=(v > 1e-12))

    # 3. Lấy Min theo hàng (axis=1)
    u = np.min(ratios, axis=1)

    return u

In [11]:
np.seterr(all='ignore') # Ignore warnings log(0)
# --- MAIN ALGORITHM ---
def one_sample_sgd_fastlog(
    A: np.ndarray, b: np.ndarray, supply_s: np.ndarray, q0: np.ndarray,
    valuations: np.ndarray, budgets: np.ndarray, p0: np.ndarray,
    lr_p=0.01, lr_q=0.01, num_iters=1000000, seed=42, log_freq=5000,
    utility_type="Linear", eps=0.00001):
    """
    Phiên bản SGD cho mô hình Ma trận A (vector q_i).
    """
    # Marks starting time of setup
    t0 = time.perf_counter()

    #--- Khởi tạo ---
    rng = np.random.default_rng(seed)
    n, m = valuations.shape
    # Copy biến dual
    p = p0.copy()
    q = q0.copy()

     # --- BƯỚC 1: KHỞI TẠO TUYỆT ĐỐI (X0 = 100) ---
    X = np.ones((n, m)) * 1
    util =  np.zeros(n)
    for j in range(n):
        if utility_type == "Linear":
            util[j] = valuations[j] @ X[j]
        elif utility_type == "Cobb-Douglas":
            util[j] = calculate_cd_utility(X[j], valuations[j])
        elif utility_type == "Leontief":
            util[j] = calculate_leo_utility(X[j], valuations[j])
        elif utility_type == "CES_8":
            util[j] = calculate_ces_utility(X[j], valuations[j], m_ces=0.5)

    # --- 2. GHI LOG TRẠNG THÁI BAN ĐẦU (t=0) ---
    # Khởi tạo lịch sử
    num_logs = num_iters // 10 + 1
    obj_hist = np.empty(num_iters)
    time_hist = np.empty(num_iters)
    log_idx = 0

    # Tính Objective tại điểm xuất phát (lúc này chưa ai mua gì)
    term_p = np.sum(p * supply_s)     # p * Supply
    term_q = np.sum(q * b)            # q * Capacity
    term_u = np.sum(budgets * np.log(util + 1e-12)) # Utility
    obj_0 = term_p + term_q + term_u - np.sum(budgets)

    obj_hist[log_idx] = obj_0
    time_hist[log_idx] = 0.0
    log_idx += 1

    print(f"--- Start SGD ({num_iters} iterations) | Initial Obj: {obj_0:.2f} ---")

    # --- WARM UP ---
    for j in range(n):
        # Dùng hàm Analytical siêu tốc
        X[j] = buyer_best_response_cvx(valuations[j], p, A, q[j], budgets[j], utility_type)

        # Cập nhật util để dùng cho vòng lặp sau
        if utility_type == "Linear":
            util[j] = valuations[j] @ X[j]
        elif utility_type == "Cobb-Douglas":
            util[j] = calculate_cd_utility(X[j], valuations[j])
        elif utility_type == "Leontief":
            util[j] = calculate_leo_utility(X[j], valuations[j])
        elif utility_type == "CES_8":
            util[j] = calculate_ces_utility(X[j], valuations[j], m_ces=0.5)

    # Tính Tổng cầu ban đầu (D) sau khi Warm up
    D = X.sum(axis=0)

    # Variables to track times for modeling parallel execution
    total_br_computation_time = 0.0 # Sum of all individual buyer_best_response_cvx times
    total_non_br_computation_time = 0.0 # Sum of all other sequential operations (gradients, dual updates)


    # Mark start of SGD loop for time measurement
    t_sgd_loop_start = time.perf_counter()

    # --- Vòng lặp SGD ---
    try:
        for t in range(1, num_iters + 1):
            # 1. Chọn buyer ngẫu nhiên
            i = int(rng.integers(n))

            # Track time for buyer_best_response_cvx (parallelizable part)
            t_br_start = time.perf_counter()
            # 2. Cập nhật D và X[i]
            D -= X[i]
            # Gọi hàm best_response MỚI (với q[i] là vector)
            x_new = buyer_best_response_cvx(valuations[i], p, A, q[i], budgets[i], utility_type)

            t_br_end = time.perf_counter()
            delta_t_br = t_br_end - t_br_start
            total_br_computation_time += delta_t_br

            X[i] = x_new
            if utility_type == "Linear":
                util[i] = valuations[i] @ X[i]
            elif utility_type == "Cobb-Douglas":
                util[i] = calculate_cd_utility(valuations[i], X[i])
            elif utility_type == "Leontief":
                util[j] = calculate_leo_utility(valuations[i], X[i])

            D += x_new

            # Track time for other sequential operations
            t_non_br_start = time.perf_counter()

            # 3. TÍNH GRADIENT
            # Gradient của p: (Tổng Cầu - Tổng Cung)
            # Nếu D > S -> g_p > 0 -> p cần tăng
            g_p = D - supply_s

            # Gradient của q_i: (Tải trọng cá nhân - Capacity cá nhân)
            # Load tại slot = A * x
            load_i = A @ x_new
            g_q_i = load_i - b[i]

            # E. CẬP NHẬT BIẾN DUAL (Gradient Ascent cho bài toán Dual)
            # Decay Learning Rate: 1 / sqrt(t)
            alpha = lr_p / np.sqrt(t)
            beta  = lr_q / np.sqrt(t)

            p = np.maximum(0, p + alpha * g_p)
            q[i] = np.maximum(0, q[i] + beta * g_q_i)

            delta_t_non_br = time.perf_counter() - t_non_br_start
            total_non_br_computation_time += delta_t_non_br

            # 6. Log định kỳ
            if t % log_freq == 0:
                # Tính Objective Value
                term_p = np.sum(p * supply_s) # Sử dụng supply_s truyền vào
                term_q = np.sum(q * b)
                term_u = np.sum(budgets * np.log(util + 1e-12))
                obj = term_p + term_q + term_u - np.sum(budgets)

                # Modelled parallel time: sum of all BR times divided by n (ideal speedup)
                # Add other non-BR times (gradients, dual updates) as sequential.
                effective_parallel_time_point = (total_br_computation_time / n) + total_non_br_computation_time

                # Update history
                obj_hist[log_idx] = obj
                time_hist[log_idx] = effective_parallel_time_point
                # Update log_idx in numpy array
                log_idx += 1
                # --- Termination conditions ---
                ratio = abs((obj_hist[log_idx - 1] - obj_hist[log_idx - 2]) / obj_hist[log_idx - 2])
                if ratio <= eps:
                    print(f"SPDS Iterations: {t}")
                    print(f"Objective value: {obj:.4f}")
                    print(f"SPDS Algorithm finsished at epsilon difference {eps} in {effective_parallel_time_point:.2f} s")
                    break
                elif ratio <= 0.0001:
                    log_freq = n * 10 / 4
                elif ratio <= 0.001:
                    log_freq = n * 10 / 2

    except KeyboardInterrupt:
        print("🛑 Interrupted by user")

    except Exception as e:
        print(f"💥 Crash: {e}")

    return obj_hist[:log_idx], time_hist[:log_idx], X.copy(), p.copy(), q.copy()

# 10 users and 24 items

In [ ]:
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_10u_24i"
print(f"--- Loading Data ---")
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path):
        valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
    print(f"✓ Valuations Shape: {valuations.shape}")
except:
    print("⚠️ Dùng dữ liệu ngẫu nhiên (Random Valuations)")
    valuations = np.random.rand(10, 24) + 0.1
valuations /= 10


# --- PARAMETERS ---
n_buyers, n_goods = valuations.shape
n_slots = 4
p0 = np.ones(n_goods)
q0 = np.zeros((n_buyers, n_slots))
np.random.seed(1)
budgets = np.array([10] * n_buyers)
supply_s = np.random.uniform(5, 10, n_goods)
capacity_b = np.random.uniform(1, 4, (n_buyers, n_slots))

# Same parameters
num_iters = 10000000
log_freq  = 250
current_lr = 0.0005
print(f"\nThiết lập hoàn tất. (n_buyers={n_buyers}, n_goods={n_goods})")

--- Loading Data ---
✓ Valuations Shape: (10, 24)

Thiết lập hoàn tất. (n_buyers=10, n_goods=24)


In [ ]:
# --- TẠO MA TRẬN A ĐỂ TRUYỀN VÀO HÀM ---
if n_goods % n_slots == 0:
    A_matrix = build_A_matrix(n_goods, n_slots)
else:
    A_matrix = np.zeros((n_slots, n_goods))
print(f"Đã tạo Ma trận A với shape: {A_matrix.shape}")

# --- OPTIMAL DUAL ALLOCATION
target_optimal_val, target_optimal_X = solve_centralized_optimal_fair(
            valuations, budgets, supply_s, capacity_b, A_matrix, utility_type="Linear")
print(f"★ Target Dual Optimal (V*): {target_optimal_val:.4f}")
if target_optimal_X is not None:
    print(f"★ Optimal Allocation Shape (X*): {target_optimal_X.shape}")

# --- RUN SPDS ---
t0 = time.perf_counter()
obj_sgd, sgd_runtime, sgd_allocation, p_new, q_new = one_sample_sgd_fastlog(
        A_matrix, capacity_b, supply_s, q0, valuations, budgets, p0,
        lr_p=current_lr, lr_q=current_lr,
        num_iters=num_iters, log_freq=log_freq)

t_sgd = time.perf_counter() - t0
print(f"\nSGD finished in {t_sgd:.2f} s")

# --- SAVE RESULT ---
all_results = {
    'objective': obj_sgd,
    'time_log': sgd_runtime,
    'alloc_final': sgd_allocation,
    'total_time': t_sgd,
    'config': {
        'lr_p': current_lr,
        'lr_q': current_lr,
        'num_iters' : len(obj_sgd) * log_freq,
        'log_freq' : log_freq
    },
    'data': {
        'n_buyers' : n_buyers,
        'n_goods' : n_goods,
        'n_slots' : n_slots,
        'budgets' : budgets,
        'capacity_b': capacity_b,
        'supply_s': supply_s,
        'p_new'      : p_new,
        'q_new'      : q_new,
        'A_matrix': A_matrix,
        'Description': "10 users data with constrained"
    }
}

# 2. Định nghĩa đường dẫn và lưu file
save_path_pickle = '/content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_10u_24i.pkl'
atomic_pickle_dump(all_results, save_path_pickle)
print(f"Đã lưu thành công TOÀN BỘ kết quả vào: {save_path_pickle}")

Đã tạo Ma trận A với shape: (4, 24)
--- Solving Centralized Fair Problem (Linear) ---
★ Target Dual Optimal (V*): -63.9093
★ Optimal Allocation Shape (X*): (10, 24)
--- Start SGD (10000000 iterations) | Initial Obj: 70.60 ---
SPDS Iterations: 270750
Objective value: -62.4956
SPDS Algorithm finsished at epsilon difference 0.0001 in 5.29 s

SGD finished in 11.57 s
[atomic_pickle_dump] Saving to temporary file in: /content/drive/MyDrive/EV_charging_project/experiment_result
[atomic_pickle_dump] ✓ Written and flushed temp file: /content/drive/MyDrive/EV_charging_project/experiment_result/tmpf64wbvdg.tmp
[atomic_pickle_dump] ✅ Successfully promoted temp file → /content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_10u_24i.pkl
Đã lưu thành công TOÀN BỘ kết quả vào: /content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_10u_24i.pkl


# 50 users and 60 items

In [ ]:
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_50u_60i"

# --- Code của bạn bắt đầu từ đây ---
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path):
        valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
    print(f"✓ Valuations Shape: {valuations.shape}")
except:
    print("⚠️ Dùng dữ liệu ngẫu nhiên (Random Valuations)")
    valuations = np.random.rand(50, 60) + 0.1
valuations /= 10


# --- PARAMETERS ---
n_buyers, n_goods = valuations.shape
n_slots = 10
p0 = np.ones(n_goods)
q0 = np.zeros((n_buyers, n_slots))
np.random.seed(1)
budgets = np.array([10] * n_buyers)
supply_s = np.random.uniform(5, 10, n_goods)
capacity_b = np.random.uniform(1, 4, (n_buyers, n_slots))

# Same parameters
num_iters = 10000000
log_freq  = 250
current_lr = 0.0005
print(f"\nThiết lập hoàn tất. (n_buyers={n_buyers}, n_goods={n_goods})")

✓ Valuations Shape: (50, 60)

Thiết lập hoàn tất. (n_buyers=50, n_goods=60)


In [ ]:
# --- TẠO MA TRẬN A ĐỂ TRUYỀN VÀO HÀM ---
if n_goods % n_slots == 0:
    A_matrix = build_A_matrix(n_goods, n_slots)
else:
    A_matrix = np.zeros((n_slots, n_goods))
print(f"Đã tạo Ma trận A với shape: {A_matrix.shape}")

# --- OPTIMAL DUAL ALLOCATION
target_optimal_val, target_optimal_X = solve_centralized_optimal_fair(
            valuations, budgets, supply_s, capacity_b, A_matrix, utility_type="Linear")
print(f"★ Target Dual Optimal (V*): {target_optimal_val:.4f}")
if target_optimal_X is not None:
    print(f"★ Optimal Allocation Shape (X*): {target_optimal_X.shape}")

# --- RUN SPDS ---
t0 = time.perf_counter()
obj_sgd, sgd_runtime, sgd_allocation, p_new, q_new = one_sample_sgd_fastlog(
        A_matrix, capacity_b, supply_s, q0, valuations, budgets, p0,
        num_iters=num_iters, lr_p=current_lr, lr_q=current_lr,
        log_freq=log_freq, eps=0.0003)

t_sgd = time.perf_counter() - t0
print(f"\nSGD finished in {t_sgd:.2f} s")

# --- SAVE RESULT ---
all_results = {
    'objective': obj_sgd,
    'time_log': sgd_runtime,
    'alloc_final': sgd_allocation,
    'total_time': t_sgd,
    'config': {
        'lr_p': current_lr,
        'lr_q': current_lr,
        'num_iters' : len(obj_sgd) * log_freq,
        'log_freq' : log_freq
    },
    'data': {
        'n_buyers' : n_buyers,
        'n_goods' : n_goods,
        'n_slots' : n_slots,
        'budgets' : budgets,
        'capacity_b': capacity_b,
        'supply_s': supply_s,
        'p0'      : p0,
        'q0'      : q0,
        'A_matrix': A_matrix,
        'Description': "50 users data with constrained"
    }
}

# 2. Định nghĩa đường dẫn và lưu file
save_path_pickle = '/content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_50u_60i.pkl'
atomic_pickle_dump(all_results, save_path_pickle)
print(f"Đã lưu thành công TOÀN BỘ kết quả vào: {save_path_pickle}")

Đã tạo Ma trận A với shape: (10, 60)
--- Solving Centralized Fair Problem (Linear) ---
★ Target Dual Optimal (V*): -960.1566
★ Optimal Allocation Shape (X*): (50, 60)
--- Start SGD (10000000 iterations) | Initial Obj: 1201.57 ---
SPDS Iterations: 325000
Objective value: -776.0213
SPDS Algorithm finsished at epsilon difference 0.0003 in 5.73 s

SGD finished in 13.43 s
[atomic_pickle_dump] Saving to temporary file in: /content/drive/MyDrive/EV_charging_project/experiment_result
[atomic_pickle_dump] ✓ Written and flushed temp file: /content/drive/MyDrive/EV_charging_project/experiment_result/tmp_muhqdlx.tmp
[atomic_pickle_dump] ✅ Successfully promoted temp file → /content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_50u_60i.pkl
Đã lưu thành công TOÀN BỘ kết quả vào: /content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_50u_60i.pkl


# 100 users and 150 items

In [12]:
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_100u_150i"

# --- Code của bạn bắt đầu từ đây ---
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path):
        valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
    print(f"✓ Valuations Shape: {valuations.shape}")
except:
    print("⚠️ Dùng dữ liệu ngẫu nhiên (Random Valuations)")
    valuations = np.random.rand(100, 150) + 0.1
valuations /= 10

# --- PARAMETERS ---
n_buyers, n_goods = valuations.shape
n_slots = 25
p0 = np.ones(n_goods)
q0 = np.zeros((n_buyers, n_slots))
np.random.seed(1)
budgets = np.array([10] * n_buyers)
supply_s = np.random.uniform(5, 10, n_goods)
capacity_b = np.random.uniform(1, 4, (n_buyers, n_slots))

# Same parameters
num_iters = 10000000
log_freq  = 10 * n_buyers
current_lr = 0.0005
print(f"\nThiết lập hoàn tất. (n_buyers={n_buyers}, n_goods={n_goods})")

✓ Valuations Shape: (100, 150)

Thiết lập hoàn tất. (n_buyers=100, n_goods=150)


In [13]:
# --- TẠO MA TRẬN A ĐỂ TRUYỀN VÀO HÀM ---
if n_goods % n_slots == 0:
    A_matrix = build_A_matrix(n_goods, n_slots)
else:
    A_matrix = np.zeros((n_slots, n_goods))
print(f"Đã tạo Ma trận A với shape: {A_matrix.shape}")

# --- OPTIMAL DUAL ALLOCATION
target_optimal_val, target_optimal_X = solve_centralized_optimal_fair(
            valuations, budgets, supply_s, capacity_b, A_matrix, utility_type="Linear")
print(f"★ Target Dual Optimal (V*): {target_optimal_val:.4f}")
if target_optimal_X is not None:
    print(f"★ Optimal Allocation Shape (X*): {target_optimal_X.shape}")

# --- RUN SPDS ---
t0 = time.perf_counter()
obj_sgd, sgd_runtime, sgd_allocation, p_new, q_new = one_sample_sgd_fastlog(
        A_matrix, capacity_b, supply_s, q0, valuations, budgets, p0,
        num_iters=num_iters, lr_p=current_lr, lr_q=current_lr,
        log_freq=log_freq)

t_sgd = time.perf_counter() - t0
print(f"\nSGD finished in {t_sgd:.2f} s")

# --- SAVE RESULT ---
all_results = {
    'objective': obj_sgd,
    'time_log': sgd_runtime,
    'alloc_final': sgd_allocation,
    'total_time': t_sgd,
    'config': {
        'lr_p': current_lr,
        'lr_q': current_lr,
        'num_iters' : len(obj_sgd) * log_freq,
        'log_freq' : log_freq
    },
    'data': {
        'n_buyers' : n_buyers,
        'n_goods' : n_goods,
        'n_slots' : n_slots,
        'budgets' : budgets,
        'capacity_b': capacity_b,
        'supply_s': supply_s,
        'p0'      : p0,
        'q0'      : q0,
        'A_matrix': A_matrix,
        'Description': "100 users data with constrained"
    }
}

# 2. Định nghĩa đường dẫn và lưu file
save_path_pickle = '/content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_100u_150i.pkl'
atomic_pickle_dump(all_results, save_path_pickle)
print(f"Đã lưu thành công TOÀN BỘ kết quả vào: {save_path_pickle}")

Đã tạo Ma trận A với shape: (25, 150)
--- Solving Centralized Fair Problem (Linear) ---
★ Target Dual Optimal (V*): -2593.4031
★ Optimal Allocation Shape (X*): (100, 150)
--- Start SGD (10000000 iterations) | Initial Obj: 110.76 ---
SPDS Iterations: 22000
Objective value: -2588.6114
SPDS Algorithm finsished at epsilon difference 1e-05 in 0.90 s

SGD finished in 2.11 s
[atomic_pickle_dump] Saving to temporary file in: /content/drive/MyDrive/EV_charging_project/experiment_result
[atomic_pickle_dump] ✓ Written and flushed temp file: /content/drive/MyDrive/EV_charging_project/experiment_result/tmpzeyh97ff.tmp
[atomic_pickle_dump] ✅ Successfully promoted temp file → /content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_100u_150i.pkl
Đã lưu thành công TOÀN BỘ kết quả vào: /content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_100u_150i.pkl


# 200 users and 240 items

In [18]:
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_200u_240i"

# --- Code của bạn bắt đầu từ đây ---
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path):
        valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
    print(f"✓ Valuations Shape: {valuations.shape}")
except:
    print("⚠️ Dùng dữ liệu ngẫu nhiên (Random Valuations)")
    valuations = np.random.rand(200, 240) + 0.1
valuations /= 10

# --- Phần còn lại của code giữ nguyên ---
n_buyers, n_goods = valuations.shape
n_slots = 40
budgets = np.full(n_buyers, 1.0)
energy  = np.ones(n_buyers)
p0      = np.ones(n_goods)
q0 = np.zeros((n_buyers, n_slots))
np.random.seed(1)
budgets = np.array([10] * n_buyers)
supply_s = np.random.uniform(5, 10, n_goods)
capacity_b = np.random.uniform(1, 4, (n_buyers, n_slots))

# Same parameters
num_iters = 10000000
log_freq  = 10 * n_buyers
current_lr = 0.0005
print(f"\nThiết lập hoàn tất. (n_buyers={n_buyers}, n_goods={n_goods})")

✓ Valuations Shape: (200, 240)

Thiết lập hoàn tất. (n_buyers=200, n_goods=240)


In [19]:
# --- TẠO MA TRẬN A ĐỂ TRUYỀN VÀO HÀM ---
if n_goods % n_slots == 0:
    A_matrix = build_A_matrix(n_goods, n_slots)
else:
    A_matrix = np.zeros((n_slots, n_goods))
print(f"Đã tạo Ma trận A với shape: {A_matrix.shape}")

# --- OPTIMAL DUAL ALLOCATION
target_optimal_val, target_optimal_X = solve_centralized_optimal_fair(
            valuations, budgets, supply_s, capacity_b, A_matrix, utility_type="Linear")
print(f"★ Target Dual Optimal (V*): {target_optimal_val:.4f}")
if target_optimal_X is not None:
    print(f"★ Optimal Allocation Shape (X*): {target_optimal_X.shape}")

# --- RUN SPDS ---
t0 = time.perf_counter()
obj_sgd, sgd_runtime, sgd_allocation, p_new, q_new = one_sample_sgd_fastlog(
        A_matrix, capacity_b, supply_s, q0, valuations, budgets, p0,
        num_iters=num_iters, lr_p=current_lr, lr_q=current_lr,
        log_freq=log_freq)

t_sgd = time.perf_counter() - t0
print(f"\nSGD finished in {t_sgd:.2f} s")

# --- SAVE RESULT ---
all_results = {
    'objective': obj_sgd,
    'time_log': sgd_runtime,
    'alloc_final': sgd_allocation,
    'total_time': t_sgd,
    'config': {
        'lr_p': current_lr,
        'lr_q': current_lr,
        'num_iters' : len(obj_sgd) * log_freq,
        'log_freq' : log_freq
    },
    'data': {
        'n_buyers' : n_buyers,
        'n_goods' : n_goods,
        'n_slots' : n_slots,
        'budgets' : budgets,
        'capacity_b': capacity_b,
        'supply_s': supply_s,
        'p0'      : p0,
        'q0'      : q0,
        'A_matrix': A_matrix,
        'Description': "200 users data with constrained"
    }
}

# 2. Định nghĩa đường dẫn và lưu file
save_path_pickle = '/content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_200u_240i.pkl'
atomic_pickle_dump(all_results, save_path_pickle)
print(f"Đã lưu thành công TOÀN BỘ kết quả vào: {save_path_pickle}")

Đã tạo Ma trận A với shape: (40, 240)
--- Solving Centralized Fair Problem (Linear) ---
★ Target Dual Optimal (V*): -6555.3234
★ Optimal Allocation Shape (X*): (200, 240)
--- Start SGD (10000000 iterations) | Initial Obj: -204.52 ---
SPDS Iterations: 23000
Objective value: -6462.2518
SPDS Algorithm finsished at epsilon difference 1e-05 in 0.43 s

SGD finished in 1.05 s
[atomic_pickle_dump] Saving to temporary file in: /content/drive/MyDrive/EV_charging_project/experiment_result
[atomic_pickle_dump] ✓ Written and flushed temp file: /content/drive/MyDrive/EV_charging_project/experiment_result/tmp649k10za.tmp
[atomic_pickle_dump] ✅ Successfully promoted temp file → /content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_200u_240i.pkl
Đã lưu thành công TOÀN BỘ kết quả vào: /content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_200u_240i.pkl


# 300 users and 360 items

In [20]:
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_300u_360i"

# --- Code của bạn bắt đầu từ đây ---
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path):
        valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
    print(f"✓ Valuations Shape: {valuations.shape}")
except:
    print("⚠️ Dùng dữ liệu ngẫu nhiên (Random Valuations)")
    valuations = np.random.rand(300, 360) + 0.1
valuations /= 10

# --- Phần còn lại của code giữ nguyên ---
n_buyers, n_goods = valuations.shape
n_slots = 60
energy  = np.ones(n_buyers)
p0      = np.ones(n_goods)
q0 = np.zeros((n_buyers, n_slots))
np.random.seed(1)
budgets = np.array([10] * n_buyers)
supply_s = np.random.uniform(5, 10, n_goods)
capacity_b = np.random.uniform(1, 4, (n_buyers, n_slots))

# Same parameters
num_iters = 10000000
log_freq  = 10 * n_buyers
current_lr = 0.0005
print(f"\nThiết lập hoàn tất. (n_buyers={n_buyers}, n_goods={n_goods})")

✓ Valuations Shape: (300, 360)

Thiết lập hoàn tất. (n_buyers=300, n_goods=360)


In [21]:
# --- TẠO MA TRẬN A ĐỂ TRUYỀN VÀO HÀM ---
if n_goods % n_slots == 0:
    A_matrix = build_A_matrix(n_goods, n_slots)
else:
    A_matrix = np.zeros((n_slots, n_goods))
print(f"Đã tạo Ma trận A với shape: {A_matrix.shape}")

# --- OPTIMAL DUAL ALLOCATION
target_optimal_val, target_optimal_X = solve_centralized_optimal_fair(
            valuations, budgets, supply_s, capacity_b, A_matrix, utility_type="Linear")
print(f"★ Target Dual Optimal (V*): {target_optimal_val:.4f}")
if target_optimal_X is not None:
    print(f"★ Optimal Allocation Shape (X*): {target_optimal_X.shape}")

# --- RUN SPDS ---
t0 = time.perf_counter()
obj_sgd, sgd_runtime, sgd_allocation, p_new, q_new  = one_sample_sgd_fastlog(
        A_matrix, capacity_b, supply_s, q0, valuations, budgets, p0,
        num_iters=num_iters, lr_p=current_lr, lr_q=current_lr,
        log_freq=log_freq)

t_sgd = time.perf_counter() - t0
print(f"\nSGD finished in {t_sgd:.2f} s")

# --- SAVE RESULT ---
all_results = {
    'objective': obj_sgd,
    'time_log': sgd_runtime,
    'alloc_final': sgd_allocation,
    'total_time': t_sgd,
    'config': {
        'lr_p': current_lr,
        'lr_q': current_lr,
        'num_iters' : len(obj_sgd) * log_freq,
        'log_freq' : log_freq
    },
    'data': {
        'n_buyers' : n_buyers,
        'n_goods' : n_goods,
        'n_slots' : n_slots,
        'budgets' : budgets,
        'capacity_b': capacity_b,
        'supply_s': supply_s,
        'p0'      : p0,
        'q0'      : q0,
        'A_matrix': A_matrix,
        'Description': "300 users data with constrained"
    }
}

# 2. Định nghĩa đường dẫn và lưu file
save_path_pickle = '/content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_300u_360i.pkl'
atomic_pickle_dump(all_results, save_path_pickle)
print(f"Đã lưu thành công TOÀN BỘ kết quả vào: {save_path_pickle}")

Đã tạo Ma trận A với shape: (60, 360)
--- Solving Centralized Fair Problem (Linear) ---
★ Target Dual Optimal (V*): -11024.8890
★ Optimal Allocation Shape (X*): (300, 360)
--- Start SGD (10000000 iterations) | Initial Obj: -278.60 ---
SPDS Iterations: 36000
Objective value: -10804.5727
SPDS Algorithm finsished at epsilon difference 1e-05 in 1.12 s

SGD finished in 2.63 s
[atomic_pickle_dump] Saving to temporary file in: /content/drive/MyDrive/EV_charging_project/experiment_result
[atomic_pickle_dump] ✓ Written and flushed temp file: /content/drive/MyDrive/EV_charging_project/experiment_result/tmp_blwi1xh.tmp
[atomic_pickle_dump] ✅ Successfully promoted temp file → /content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_300u_360i.pkl
Đã lưu thành công TOÀN BỘ kết quả vào: /content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_300u_360i.pkl


# 400 users and 480 items

In [22]:
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_400u_480i"

# --- Code của bạn bắt đầu từ đây ---
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path):
        valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
    print(f"✓ Valuations Shape: {valuations.shape}")
except:
    print("⚠️ Dùng dữ liệu ngẫu nhiên (Random Valuations)")
    valuations = np.random.rand(400, 480) + 0.1
valuations /= 10

# --- Phần còn lại của code giữ nguyên ---
n_buyers, n_goods = valuations.shape
n_slots = 80
energy  = np.ones(n_buyers)
p0      = np.ones(n_goods)
q0 = np.zeros((n_buyers, n_slots))
np.random.seed(1)
budgets = np.array([10] * n_buyers)
supply_s = np.random.uniform(5, 10, n_goods)
capacity_b = np.random.uniform(1, 4, (n_buyers, n_slots))

# Same parameters
num_iters = 10000000
log_freq  = 10 * n_buyers
current_lr = 0.0005
print(f"\nThiết lập hoàn tất. (n_buyers={n_buyers}, n_goods={n_goods})")

✓ Valuations Shape: (400, 480)

Thiết lập hoàn tất. (n_buyers=400, n_goods=480)


In [23]:
# --- TẠO MA TRẬN A ĐỂ TRUYỀN VÀO HÀM ---
if n_goods % n_slots == 0:
    A_matrix = build_A_matrix(n_goods, n_slots)
else:
    A_matrix = np.zeros((n_slots, n_goods))
print(f"Đã tạo Ma trận A với shape: {A_matrix.shape}")

# --- OPTIMAL DUAL ALLOCATION
target_optimal_val, target_optimal_X = solve_centralized_optimal_fair(
            valuations, budgets, supply_s, capacity_b, A_matrix, utility_type="Linear")
print(f"★ Target Dual Optimal (V*): {target_optimal_val:.4f}")
if target_optimal_X is not None:
    print(f"★ Optimal Allocation Shape (X*): {target_optimal_X.shape}")

# --- RUN SPDS ---
t0 = time.perf_counter()
obj_sgd, sgd_runtime, sgd_allocation, p_new, q_new = one_sample_sgd_fastlog(
        A_matrix, capacity_b, supply_s, q0, valuations, budgets, p0,
        num_iters=num_iters, lr_p=current_lr, lr_q=current_lr,
        log_freq=log_freq)

t_sgd = time.perf_counter() - t0
print(f"\nSGD finished in {t_sgd:.2f} s")

# --- SAVE RESULT ---
all_results = {
    'objective': obj_sgd,
    'time_log': sgd_runtime,
    'alloc_final': sgd_allocation,
    'total_time': t_sgd,
    'config': {
        'lr_p': current_lr,
        'lr_q': current_lr,
        'num_iters' : len(obj_sgd) * log_freq,
        'log_freq' : log_freq
    },
    'data': {
        'n_buyers' : n_buyers,
        'n_goods' : n_goods,
        'n_slots' : n_slots,
        'budgets' : budgets,
        'capacity_b': capacity_b,
        'supply_s': supply_s,
        'p_new'      : p_new,
        'q_new'      : q_new,
        'A_matrix': A_matrix,
        'Description': "300 users data with constrained"
    }
}

# 2. Định nghĩa đường dẫn và lưu file
save_path_pickle = '/content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_400u_480i.pkl'
atomic_pickle_dump(all_results, save_path_pickle)
print(f"Đã lưu thành công TOÀN BỘ kết quả vào: {save_path_pickle}")

Đã tạo Ma trận A với shape: (80, 480)
--- Solving Centralized Fair Problem (Linear) ---
★ Target Dual Optimal (V*): -15871.5403
★ Optimal Allocation Shape (X*): (400, 480)
--- Start SGD (10000000 iterations) | Initial Obj: -384.52 ---
SPDS Iterations: 48000
Objective value: -15302.0717
SPDS Algorithm finsished at epsilon difference 1e-05 in 1.76 s

SGD finished in 4.13 s
[atomic_pickle_dump] Saving to temporary file in: /content/drive/MyDrive/EV_charging_project/experiment_result
[atomic_pickle_dump] ✓ Written and flushed temp file: /content/drive/MyDrive/EV_charging_project/experiment_result/tmp6fmwc0n5.tmp
[atomic_pickle_dump] ✅ Successfully promoted temp file → /content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_400u_480i.pkl
Đã lưu thành công TOÀN BỘ kết quả vào: /content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_400u_480i.pkl


# 500 users and 600 items

In [24]:
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_500u_600i"

# --- Code của bạn bắt đầu từ đây ---
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path):
        valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
    print(f"✓ Valuations Shape: {valuations.shape}")
except:
    print("⚠️ Dùng dữ liệu ngẫu nhiên (Random Valuations)")
    valuations = np.random.rand(500, 600) + 0.1
valuations /= 10

# --- Phần còn lại của code giữ nguyên ---
n_buyers, n_goods = valuations.shape
n_slots = 100
energy  = np.ones(n_buyers)
p0      = np.ones(n_goods)
q0 = np.zeros((n_buyers, n_slots))
np.random.seed(1)
budgets = np.array([10] * n_buyers)
supply_s = np.random.uniform(5, 10, n_goods)
capacity_b = np.random.uniform(1, 4, (n_buyers, n_slots))

# Same parameters
num_iters = 10000000
log_freq  = 10 * n_buyers
current_lr = 0.0005
print(f"\nThiết lập hoàn tất. (n_buyers={n_buyers}, n_goods={n_goods})")

✓ Valuations Shape: (500, 600)

Thiết lập hoàn tất. (n_buyers=500, n_goods=600)


In [25]:
# --- TẠO MA TRẬN A ĐỂ TRUYỀN VÀO HÀM ---
if n_goods % n_slots == 0:
    A_matrix = build_A_matrix(n_goods, n_slots)
else:
    A_matrix = np.zeros((n_slots, n_goods))
print(f"Đã tạo Ma trận A với shape: {A_matrix.shape}")

# --- OPTIMAL DUAL ALLOCATION
target_optimal_val, target_optimal_X = solve_centralized_optimal_fair(
            valuations, budgets, supply_s, capacity_b, A_matrix, utility_type="Linear")
print(f"★ Target Dual Optimal (V*): {target_optimal_val:.4f}")
if target_optimal_X is not None:
    print(f"★ Optimal Allocation Shape (X*): {target_optimal_X.shape}")

# --- RUN SPDS ---
t0 = time.perf_counter()
obj_sgd, sgd_runtime, sgd_allocation, p_new, q_new = one_sample_sgd_fastlog(
        A_matrix, capacity_b, supply_s, q0, valuations, budgets, p0,
        num_iters=num_iters, lr_p=current_lr, lr_q=current_lr,
        log_freq=log_freq)

t_sgd = time.perf_counter() - t0
print(f"\nSGD finished in {t_sgd:.2f} s")

# --- SAVE RESULT ---
all_results = {
    'objective': obj_sgd,
    'time_log': sgd_runtime,
    'alloc_final': sgd_allocation,
    'total_time': t_sgd,
    'config': {
        'lr_p': current_lr,
        'lr_q': current_lr,
        'num_iters' : len(obj_sgd) * log_freq,
        'log_freq' : log_freq
    },
    'data': {
        'n_buyers' : n_buyers,
        'n_goods' : n_goods,
        'n_slots' : n_slots,
        'budgets' : budgets,
        'capacity_b': capacity_b,
        'supply_s': supply_s,
        'p_new'      : p_new,
        'q_new'      : q_new,
        'A_matrix': A_matrix,
        'Description': "300 users data with constrained"
    }
}

# 2. Định nghĩa đường dẫn và lưu file
save_path_pickle = '/content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_500u_600i.pkl'
atomic_pickle_dump(all_results, save_path_pickle)
print(f"Đã lưu thành công TOÀN BỘ kết quả vào: {save_path_pickle}")

Đã tạo Ma trận A với shape: (100, 600)
--- Solving Centralized Fair Problem (Linear) ---
★ Target Dual Optimal (V*): -20971.4591
★ Optimal Allocation Shape (X*): (500, 600)
--- Start SGD (10000000 iterations) | Initial Obj: -491.74 ---
SPDS Iterations: 65000
Objective value: -20421.7793
SPDS Algorithm finsished at epsilon difference 1e-05 in 2.33 s

SGD finished in 5.72 s
[atomic_pickle_dump] Saving to temporary file in: /content/drive/MyDrive/EV_charging_project/experiment_result
[atomic_pickle_dump] ✓ Written and flushed temp file: /content/drive/MyDrive/EV_charging_project/experiment_result/tmpyt87v50u.tmp
[atomic_pickle_dump] ✅ Successfully promoted temp file → /content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_500u_600i.pkl
Đã lưu thành công TOÀN BỘ kết quả vào: /content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_500u_600i.pkl


# 600 users and 720 items

In [26]:
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_600u_720i"

# --- Code của bạn bắt đầu từ đây ---
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path):
        valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
    print(f"✓ Valuations Shape: {valuations.shape}")
except:
    print("⚠️ Dùng dữ liệu ngẫu nhiên (Random Valuations)")
    valuations = np.random.rand(600, 720) + 0.1
valuations /= 10

# --- Phần còn lại của code giữ nguyên ---
n_buyers, n_goods = valuations.shape
n_slots = 120
energy  = np.ones(n_buyers)
p0      = np.ones(n_goods)
q0 = np.zeros((n_buyers, n_slots))
np.random.seed(1)
budgets = np.array([10] * n_buyers)
supply_s = np.random.uniform(5, 10, n_goods)
capacity_b = np.random.uniform(1, 4, (n_buyers, n_slots))

# Same parameters
num_iters = 10000000
log_freq  = 10 * n_buyers
current_lr = 0.0005
print(f"\nThiết lập hoàn tất. (n_buyers={n_buyers}, n_goods={n_goods})")

⚠️ Dùng dữ liệu ngẫu nhiên (Random Valuations)

Thiết lập hoàn tất. (n_buyers=600, n_goods=720)


In [27]:
# --- TẠO MA TRẬN A ĐỂ TRUYỀN VÀO HÀM ---
if n_goods % n_slots == 0:
    A_matrix = build_A_matrix(n_goods, n_slots)
else:
    A_matrix = np.zeros((n_slots, n_goods))
print(f"Đã tạo Ma trận A với shape: {A_matrix.shape}")

# --- OPTIMAL DUAL ALLOCATION
target_optimal_val, target_optimal_X = solve_centralized_optimal_fair(
            valuations, budgets, supply_s, capacity_b, A_matrix, utility_type="Linear")
print(f"★ Target Dual Optimal (V*): {target_optimal_val:.4f}")
if target_optimal_X is not None:
    print(f"★ Optimal Allocation Shape (X*): {target_optimal_X.shape}")

# --- RUN SPDS ---
t0 = time.perf_counter()
obj_sgd, sgd_runtime, sgd_allocation, p_new, q_new = one_sample_sgd_fastlog(
        A_matrix, capacity_b, supply_s, q0, valuations, budgets, p0,
        num_iters=num_iters, lr_p=current_lr, lr_q=current_lr,
        log_freq=log_freq)

t_sgd = time.perf_counter() - t0
print(f"\nSGD finished in {t_sgd:.2f} s")

# --- SAVE RESULT ---
all_results = {
    'objective': obj_sgd,
    'time_log': sgd_runtime,
    'alloc_final': sgd_allocation,
    'total_time': t_sgd,
    'config': {
        'lr_p': current_lr,
        'lr_q': current_lr,
        'num_iters' : len(obj_sgd) * log_freq,
        'log_freq' : log_freq
    },
    'data': {
        'n_buyers' : n_buyers,
        'n_goods' : n_goods,
        'n_slots' : n_slots,
        'budgets' : budgets,
        'capacity_b': capacity_b,
        'supply_s': supply_s,
        'p_new'      : p_new,
        'q_new'      : q_new,
        'A_matrix': A_matrix,
        'Description': "300 users data with constrained"
    }
}

# 2. Định nghĩa đường dẫn và lưu file
save_path_pickle = '/content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_600u_720i.pkl'
atomic_pickle_dump(all_results, save_path_pickle)
print(f"Đã lưu thành công TOÀN BỘ kết quả vào: {save_path_pickle}")

Đã tạo Ma trận A với shape: (120, 720)
--- Solving Centralized Fair Problem (Linear) ---
★ Target Dual Optimal (V*): -79.7667
★ Optimal Allocation Shape (X*): (600, 720)
--- Start SGD (10000000 iterations) | Initial Obj: 21996.63 ---
SPDS Iterations: 1953000
Objective value: -70.6667
SPDS Algorithm finsished at epsilon difference 1e-05 in 83.97 s

SGD finished in 195.04 s
[atomic_pickle_dump] Saving to temporary file in: /content/drive/MyDrive/EV_charging_project/experiment_result
[atomic_pickle_dump] ✓ Written and flushed temp file: /content/drive/MyDrive/EV_charging_project/experiment_result/tmpldhhj1fz.tmp
[atomic_pickle_dump] ✅ Successfully promoted temp file → /content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_600u_720i.pkl
Đã lưu thành công TOÀN BỘ kết quả vào: /content/drive/MyDrive/EV_charging_project/experiment_result/spds_results_600u_720i.pkl
